# Knitting


In [ ]:
import math
import numpy as np
from numpy.typing import NDArray

arr = np.array

In [ ]:
import pyvista as pv
from matplotlib.cm import tab10

# pv.set_jupyter_backend("server")
tab10colors = tab10.colors  # type: ignore
yrnclr = [tab10.colors[4], tab10.colors[6]]

In [ ]:
%%html
<style>
    :root {
        --jp-content-font-color0: var(--vscode-editor-foreground);
        --jp-content-font-color1: var(--vscode-editor-foreground);
        --jp-widgets-color: var(--vscode-editor-foreground);
        --jp-widgets-input-color: var(--vscode-editor-foreground);
        --jp-widgets-input-background-color: var(--vscode-editor-background);
        --jp-widgets-font-size: var(--vscode-editor-font-size);
    }
    .jupyter-widgets input {
        background-color: var(--jp-widgets-input-background-color);
    }
    .cell-output-ipywidget-background {
        background-color: transparent !important;
    }
</style>


## Structure

Axes:

- $U$: crosswise direction of courses, left to right, texture horizontel
- $V$: lengthwise diretion of wales, bottom to top, texture vertical
- $W$: facewise diretion of bulging, back to front, texture bump/normal

In plain dense knitting (without skipping, twisting, stretching, tucking) there are 2 basic stitch types:

- `\/`: (лицевая) knit stich, moving back to front, legs of the loop are visible on front
- `<>`: (изнаночная) purl stitch, moving front to back, head of the spporting loop is visible on front

Stitches are placed in a square/rectangular grid, and their types are defined by design of a pattern.


Every loop makes 4 intersections with loops neighbouting in its wale.

In the order along yarn:

- $K^{bl}$: bottom left (leg) with top left (head) of one below
- $K^{tl}$: top left (head) with bottom left (leg) of one above
- $K^{tr}$: top right (head) with bottom left (leg) of one above
- $K^{br}$: bottom right (leg) with top right (head) of one below

Because of thickness, each knot is actually 2 points of contact. That forms an axis of knot under some angle to main axes.

> Apparently, in normal conditions the angle is constant across whole fabric.

In the knot both yarns have some tangential direction: $K = \big[ \delta u, \delta v, \delta w]$

Directions of yarn at the knots for current legs and supporting head:

- `\`: $K^{bl}|_{curr} = [+\alpha, +\beta, +\theta], K^{tl}|_{supp} = [+\alpha, +\beta, -\theta]$
- `/`: $K^{br}|_{curr} = [+\alpha, -\beta, -\theta], K^{tr}|_{supp} = [+\alpha, -\beta, +\theta]$
- `<`: $K^{bl}|_{curr} = [+\alpha, +\beta, -\theta], K^{tl}|_{supp} = [+\alpha, +\beta, +\theta]$
- `>`: $K^{br}|_{curr} = [+\alpha, -\beta, +\theta], K^{tr}|_{supp} = [+\alpha, -\beta, -\theta]$

With some particular/symmetric angle of knots $\phi$, $\alpha = cos \phi,  \beta = sin \phi$ (or vice versa)

The $\theta$ is some thickness-dependant aspect of knot configuration. Stitch type determines direction/sign of it.

Everything is symmetrical all the way:

- $\delta w |_{supp} = - \delta w|_{curr}$
- $\delta w|^{l} = -\delta w|^{r}$


In [ ]:
xpoints = arr([[0.0, 1.0, 0.0], [0.0, 2.0, 0.0], [1.0, 2.0, 0.0], [1.0, 1.0, 0.0]])

draft = arr([
    [-1.0 + 0.125, 1.0 - 0.125, 0.0],
    [0.0 - 0.125, 1.0 - 0.125, 0.0],
    [0.0 + 0.125, 1.0 + 0.125, 0.0],
    [0.0 - 0.125, 2.0 - 0.125, 0.0],
    [0.0 + 0.125, 2.0 + 0.125, 0.0],
    [1.0 - 0.125, 2.0 + 0.125, 0.0],
    [1.0 + 0.125, 2.0 - 0.125, 0.0],
    [1.0 - 0.125, 1.0 + 0.125, 0.0],
    [1.0 + 0.125, 1.0 - 0.125, 0.0],
    [2.0 - 0.125, 1.0 - 0.125, 0.0],
])

opoints = arr([[-0.5 + 0.125, 1.0 - 0.125, 0], [0, 1.5, 0], [0.5, 2.125, 0], [1, 1.5, 0], [1.5 - 0.125, 1.0 - 0.125, 0]])

In [ ]:
THETA = math.pi / 3.0  # angle of knots
BANG = 0.25  # velocity at knots

xtang = arr([[1, 1, 0], [1, 1, 0], [1, -1, 0], [1, -1, 0]]) * arr([math.cos(THETA), math.sin(THETA), 0])
xbang = xtang * BANG

In [ ]:
pl1 = pv.Plotter()
pl1.set_background(pv.Color("#303030"))
_ = pl1.add_mesh(pv.Spline(draft, 100), name="draft", line_width=32, opacity=0.125)

pl1.add_points(xpoints, color="black", point_size=8, render_points_as_spheres=True)
pl1.add_point_labels(xpoints + arr([0, 0, 0.2]), ("bl", "tl", "tr", "br"), shape=None, text_color="white", point_color="black")
pl1.add_arrows(xpoints, xbang, color="black")
pl1.add_point_labels(opoints + arr([0, 0, 0.2]), ["butt", "leg_l", "head", "leg_r", "butt"], shape=None, point_color="gray", point_size=None, text_color="gray")
pl1.view_xy()
pl1.show()

#### Yarn


In [ ]:
from splines import megacurve


def spline(points):
    return megacurve(2, points, np.arange(0.0, 1.0, 0.125))

In [ ]:
xxtang = np.repeat(xtang, 2, axis=0)
xxbang = np.repeat(xbang, 2, axis=0)
xxbang[::2] *= -1
controls = np.repeat(xpoints, 2, axis=0) + xxbang
curve = spline(np.concat([controls + arr([-2.0, 0, 0]), controls, controls + arr([2.0, 0, 0])]))

In [ ]:
pl2 = pv.Plotter()
pl2.set_background(pv.Color("#303030"))
stuff = (
    pl2.add_points(controls, color="gray", point_size=8, render_points_as_spheres=True),
    pl2.add_mesh(pv.lines_from_points(controls), line_width=2, color="gray"),
    pl2.add_mesh(pv.lines_from_points(curve + arr([0, -1, 0])), color=yrnclr[0], line_width=4),
    pl2.add_mesh(pv.lines_from_points(curve + arr([0, +1, 0])), color=yrnclr[0], line_width=4),
    pl2.add_mesh(pv.lines_from_points(curve), color=yrnclr[1], line_width=4),
)

pl2.view_xy()
pl2.show()

#### Aparteid


In [ ]:
DIST = 0.1  # amount of pull apart, in orthogonal direction

rotl = arr([[0, 1, 0], [-1, 0, 0], [0, 0, 1]])
rotr = arr([[0, -1, 0], [1, 0, 0], [0, 0, 1]])

# i dunno
tangpart = np.array([tang @ rot for tang, rot in zip(xxtang, (rotr, rotr, rotl, rotl, rotl, rotl, rotr, rotr))])
contradj = controls + tangpart * DIST
curvadj = spline(np.concat([contradj + arr([-2.0, 0, 0]), contradj, contradj + arr([2.0, 0, 0])]))

In [ ]:
_ = pl2.add_arrows(controls, tangpart, mag=DIST, name="shift", color="black")

In [ ]:
stuffadj = (
    pl2.add_points(contradj, color="gray", point_size=8, render_points_as_spheres=True),
    pl2.add_mesh(pv.lines_from_points(contradj), line_width=2, color="gray"),
    pl2.add_mesh(pv.lines_from_points(curvadj + arr([0, -1, 0])), color=yrnclr[0], line_width=4),
    pl2.add_mesh(pv.lines_from_points(curvadj + arr([0, +1, 0])), color=yrnclr[0], line_width=4),
    pl2.add_mesh(pv.lines_from_points(curvadj), color=yrnclr[1], line_width=4),
)

In [ ]:
def toggle_stuff(flag):
    for it in stuff:
        it.SetVisibility(not flag)
    for it in stuffadj:
        it.SetVisibility(flag)


pl2.add_checkbox_button_widget(toggle_stuff)
pl2.render()

## Patterns

Patterns determine types of stitches in each course/wale, the stitches correspond to pairs of knots for legs of loops.
Types of head knots are symmetric to leg knots of a loop above.

- `\/`: the knit stich: $\big[ K^{bl}, K^{br} \big] = \big[ [+\alpha, +\beta, +\theta], [+\alpha, -\beta, -\theta] \big]$
- `<>`: the purl stitch: $\big[ K^{bl}, K^{br} \big] = \big[ [+\alpha, +\beta, -\theta], [+\alpha, -\beta, +\theta] \big]$

The $\alpha, \beta, \theta$ are some paramaters of fabric. Signs of $\delta u, \delta v$ are detemined by knots roles/orientation.

Pattern only affect direction/sign of the $\delta w$


#### Stockinette / Лицевая гладь

Plain knitting

```
\/\/
\/\/
```

Reverse / Изнаночная

```
<><>
<><>
```


In [ ]:
stock = arr([[+1, -1]])
stockrev = arr([[-1, +1]])

#### Garter / Платочная

```
<><>
\/\/
<><>
\/\/
```


In [ ]:
garter = arr([
    [+1, -1],
    [-1, +1],
])

### Rib / Резинка

```
\/<>\/<>
\/<>\/<>
```


In [ ]:
rib = arr([[+1, -1, -1, +1]])

### Moss

```
\/<>\/<>
<>\/<>\/
\/<>\/<>
```


In [ ]:
moss = arr([
    [+1, -1, -1, +1],
    [-1, +1, +1, -1],
])

### Basketweave / Корзинка

```
\/\/<><>
\/\/<><>
<><>\/\/
<><>\/\/
```


In [ ]:
basket = arr([
    [-1, +1, -1, +1, +1, -1, +1, -1],
    [-1, +1, -1, +1, +1, -1, +1, -1],
    [+1, -1, +1, -1, -1, +1, -1, +1],
    [+1, -1, +1, -1, -1, +1, -1, +1],
])

basket3 = arr([
    [-1, +1, -1, +1, -1, +1, +1, -1, +1, -1, +1, -1],
    [-1, +1, -1, +1, -1, +1, +1, -1, +1, -1, +1, -1],
    [-1, +1, -1, +1, -1, +1, +1, -1, +1, -1, +1, -1],
    [+1, -1, +1, -1, +1, -1, -1, +1, -1, +1, -1, +1],
    [+1, -1, +1, -1, +1, -1, -1, +1, -1, +1, -1, +1],
    [+1, -1, +1, -1, +1, -1, -1, +1, -1, +1, -1, +1],
])

## Sampling

A single loop goes through 4 knots in adjacent courses and wales:

- $K^{bl}: u_0, v_0$ ­— edge knot
- $K^{tl} = \~K^{bl}|_{above}: u_0, v_0 + 1$
- $K^{tr} = \~K^{br}|_{above}: u_0 + 1, v_0 + 1$
- $K^{br}: u_0+1, v_0$
- $K^{bl}|_{next}: u_0+2, v_0$

Reconstructing from pattern $P$, unscaled:

- $K^{bl} = [+1, +1, + P_{u_0, v_0}]$
- $K^{tl} = [+1, +1, - P_{u_0, v_0+1}]$
- $K^{tr} = [+1, -1, - P_{u_0+1, v_0+1}]$
- $K^{br} = [+1, -1, + P_{u_0+1, v_0}]$


In [ ]:
# I dunno how to vectorize
def sample(pattern: NDArray, v0: int, u0: int):
    """Get tangent vectors for 4 anchor knot points
    for a loop starting at v0, u0
    [botl, topl, topr, botr]
    """
    vdim, udim = pattern.shape
    return arr([
        [+1, +1, pattern[v0 % vdim, u0 % udim]],
        [+1, +1, -pattern[(v0 + 1) % vdim, u0 % udim]],
        [+1, -1, -pattern[(v0 + 1) % vdim, (u0 + 1) % udim]],
        [+1, -1, pattern[v0 % vdim, (u0 + 1) % udim]],
    ])

## Curving


In [ ]:
from splines import megacurve


def spline(points):
    return megacurve(3, points, np.arange(0.0, 1.0, 0.125))

In [ ]:
THETA = math.pi / 3.0  # angle of knot major axis
THICK = 0.25  # thickness of something
XTANGSCALE = arr([math.cos(THETA), math.sin(THETA), THICK])  # scale of knot tangets
BANG = arr([0.25, 0.25, 1])  # shift along xtang
DIST = THICK

In [ ]:
xpoints = arr([[0.0, 1.0, 0.0], [0.0, 2.0, 0.0], [1.0, 2.0, 0.0], [1.0, 1.0, 0.0]])
controls0 = np.repeat(xpoints, 2, axis=0)  # 2 cpoints per 1 xpoint

In [ ]:
pattern = basket3
vdim, udim = pattern.shape
vsize, usize = 6, 12

In [ ]:
rotl = arr([[0, 1, 0], [-1, 0, 0], [0, 0, 1]])
rotr = arr([[0, -1, 0], [1, 0, 0], [0, 0, 1]])


def sample_loop(v: int, u: int):
    tangs = sample(pattern, v, u) * XTANGSCALE
    norms = arr([t @ r for t, r in zip(tangs, (rotr, rotl, rotl, rotr))])
    bangs = np.repeat(tangs, 2, axis=0) * BANG
    bangs[::2] *= -1
    sides = np.repeat(norms, 2, axis=0) * DIST
    return controls0 + bangs + sides


def sample_course(v: int):
    return np.concat([sample_loop(v, u) + arr([u, v, 0]) for u in range(0, usize - 1, 2)])

In [ ]:
controlss = [sample_course(v) for v in range(vsize)]
curves = [spline(ctrl) for ctrl in controlss]

In [ ]:
pl = pv.Plotter()
pl.set_background(pv.Color("#303030"))
_ = [pl.add_mesh(pv.lines_from_points(c)) for c in controlss]
_ = [pl.add_mesh(pv.Spline(c).tube(radius=0.875 * THICK, n_sides=12), smooth_shading=True, color=yrnclr[i % 2]) for i, c in enumerate(curves)]
pl.view_xy()
pl.show()